# Step 26 — cross-platform: the genes all three studies measure

**Data types: RNA_array** (GSE65391, sites A, B, C), **bulk_RNA_seq** (GSE232381), **scRNA_seq
pseudobulk** (GSE135779). **Reads:** `step10_site_*`, `step13_model.rds`, `step23_GSE232381.rds`,
`step25_GSE135779.rds`. **Writes:** `step26_shared.rds`.

**How a microarray study and RNA sequencing studies are used together.**

| | GSE65391 | GSE232381 | GSE135779 |
|---|---|---|---|
| data type | RNA_array | bulk_RNA_seq | scRNA_seq, as pseudobulk |
| tissue | whole blood | peripheral blood cells | PBMC |
| values | log2 intensity | log2 counts per million (TMM, voom) | log2 counts per million (TMM, voom) |
| healthy children | 46 | none | 11 |

1. **Never in one matrix.** Array intensities and sequencing counts differ in scale, noise and dynamic
   range. Each study is its own site and keeps its own values.
2. **Shared genes.** All three are keyed to the same NCBI gene symbols (steps 02, 23, 25).
3. **Shared units.** Each study standardises every gene against its own reference: its healthy
   children where it has them, otherwise all its samples. "Two reference SDs above the reference mean"
   means the same thing on every platform.
4. **Scale-free statistics only.** The median-z interferon score, and correlation to centroids.

What travels between studies is a gene list, per-gene summaries, centroids and counts, as between the
simulated sites.

**Platform, tissue and cohort are the same thing** for each study and cannot be separated. That limit
is stated wherever a comparison depends on it.

In [1]:
source("../src/paths.R")
source("../src/federation.R")
start_log("26")
model <- readRDS(art("step13_model.rds"))
genes_of <- list(
  GSE65391  = rownames(readRDS(site_file("10", "A"))$E),       # every site carries the same genes
  GSE232381 = rownames(readRDS(art("step23_GSE232381.rds"))$E),
  GSE135779 = rownames(readRDS(art("step25_GSE135779.rds"))$E))
sapply(genes_of, length)
shared <- Reduce(intersect, genes_of)
length(shared)

GSE65391 GSE232381 GSE135779 
    27984     14692     14409

[1] 11469

## Coverage of what the model needs

In [2]:
ce <- model$centroids
markers <- unlist(lapply(rownames(ce), function(e) {
  d <- ce[e, ] - colMeans(ce[rownames(ce) != e, , drop = FALSE]); names(sort(d, decreasing = TRUE))[1:30] }))
ifn6 <- read_gene_set("ifn-type1-6.txt")
cov <- sapply(genes_of[-1], function(g) c(
  model_genes_1000 = mean(model$genes %in% g), markers_60 = mean(markers %in% g), ifn6 = mean(ifn6 %in% g)))
round(cov, 3)
c(model_genes_in_all_three = sum(model$genes %in% shared))

,GSE232381,GSE135779
model_genes_1000,0.776,0.681
markers_60,0.933,0.650
ifn6,1.000,1.000


model_genes_in_all_three 
                     677

The transfer in step 29 refuses to run below 70% coverage of the model genes.

In [3]:
setdiff(markers, shared)                         # marker genes some study does not measure
saveRDS(list(shared = shared, markers = markers, coverage = cov), art("step26_shared.rds"))

[1] "HNRNPA1P7"  "HNRNPA1P10" "LOC652694"  "SERPINA10"  "NLRC3"     
 [6] "CEACAM6"    "LOC653600"  "LTF"        "DEFA4"      "OLFM4"     
[11] "ARG1"       "CEACAM8"    "DEFA1B"     "TCN1"       "DEFA3"     
[16] "RAP1GAP"    "ORM1"       "PGLYRP1"    "CTSG"       "GYPE"      
[21] "RUNDC3A"

## Findings

11,469 genes are measured in all three studies. GSE232381 covers 78% of the model's 1,000 genes and
GSE135779 only 68%, below the 70% the transfer requires. **The genes missing from GSE135779 are mostly
the granulocyte genes that mark E2** (LTF, DEFA3, DEFA4, CEACAM8, CTSG, OLFM4, ARG1 and others). In a
PBMC preparation they are too rarely counted to pass `filterByExpr`. Of the 60 marker genes, 39 are
measured everywhere: 25 E1 markers and 14 E2 markers. All six interferon genes are measured everywhere.